In [1]:
import os

import click
import mlflow
from dotenv import load_dotenv
from mlflow.genai import evaluate
from mlflow.genai.optimize import GepaPromptOptimizer
from Evaluating_prompt_optimization_techniques_for_water_management_LLM_assistant_with_RAG.text2sql.core import (
    format_schema_for_prompt,
    load_schema,
)
from experiments.text2sql.harness import (
    ENDPOINTS,
    build_sql_judge_scorer,
    create_predict_fn,
    load_dataset,
    log_global_params,
    make_run_name,
    setup_mlflow,
)

/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
questions_path = '/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/data/text2sql/deflated_75_sqls.json'
schema_path = '/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/src/Evaluating_prompt_optimization_techniques_for_water_management_LLM_assistant_with_RAG/tenants/green_roof/sensordata.py'

model = 'openai/alias-eve'
endpoint = 'kisski'

teacher_model = 'openai:/alias-eve'

judge_model = model
judge_endpoint = endpoint

db_path = '/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/data/water.duckdb'

In [3]:
load_dotenv()
experiment_name = 'train-text2sql'
if not experiment_name:
    raise click.ClickException("MLFLOW_EVAL_EXPERIMENT_NAME is not set in the environment.")
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
setup_mlflow(tracking_uri, experiment_name)
schema_text = format_schema_for_prompt(load_schema(schema_path))
data = load_dataset(questions_path)
predict_fn = create_predict_fn(model, endpoint, schema_text)
judge_scorer = build_sql_judge_scorer(
    judge_model, judge_endpoint, schema_text, db_path
)

In [ ]:
mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=data,
    prompt_uris=['prompts:/text2sql_system/2'],
    optimizer=GepaPromptOptimizer(       
        reflection_model=teacher_model,
        max_metric_calls=9,
        display_progress_bar=True,
    ),
    scorers=[judge_scorer],
    enable_tracking=True
    )

2026/06/12 14:48:34 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/12 14:48:34 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
GEPA Optimization:   0%|          | 0/9 [00:00<?, ?rollouts/s]